# Sprint 8 - Dual-head model (eliminate catastrophic forgetting)

**Why this sprint:** Sprint 7 proved the lab-vs-field trade-off is structural across 3
backbones. No single shared head can stay good at both domains because mixed training
overwrites the head on whichever domain dominates each epoch. The fix: two independent
heads on the same backbone, each trained on its own domain.

**Architecture:**
```
backbone (shared, frozen)
  ├── head_lab   -> trained on PlantVillage only
  └── head_field -> trained on PlantDoc only
```

**Inference:** run both heads, return the higher-confidence prediction per sample.

**Sprint 8 gates (must BOTH pass on ONE checkpoint):**
1. **Field:** PlantDoc F1 >= 0.60 (stretch 0.70)
2. **Lab:** PlantVillage F1 >= 0.85

### Where we are (Sprint 7 results, in the CSV)
- baseline (MobileNetV2): PlantVillage 0.9501 F1 | PlantDoc 0.1116 F1
- `both_resnet50` (v11, field-strong): PlantDoc **0.6554** F1 but forgot lab (0.2857)
- `both_efficientnet` (v12, field-strong): PlantDoc 0.5719 F1 | PlantVillage 0.4541 F1
- `mixed` (MobileNetV2): PlantVillage **0.9592** F1 | PlantDoc 0.4107 F1
- `mixed_from_field_resnet50` (v13): PlantVillage 0.9589 F1 | PlantDoc 0.4238 F1
- `mixed_from_field_resnet50_x8` (v13_x8): PlantVillage 0.9508 F1 | PlantDoc 0.4709 F1

**The structural finding:** no single shared head clears both gates. Every backbone
splits into field-strong (high field, catastrophic lab forgetting) or lab-strong
(high lab, field collapses back to ~0.42). Dual-head is the architectural fix.

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Same as Sprint 7: Drive holds the data archives and checkpoints.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label):
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as every sprint: unzip both archives, build train/val/test folders (seed 42,
deterministic) plus class_map.json. Idempotent.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)

## Step 4 - Train head_lab on PlantVillage (dual-head Stage 1)

Train only the lab head on PlantVillage. The backbone and field head stay frozen.

Artifacts: `best_plantvillage_dual_head_lab.pt` (on Drive).

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "lab",
    "--tag", "dual_head_lab",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head lab training failed")
print("Done. Lab head trained.")

## Step 5 - Train head_field on PlantDoc (dual-head Stage 2)

Load the dual-head checkpoint from Step 4. Freeze everything except head_field.
Train on PlantDoc mapped to the PlantVillage label space.

Artifacts: `best_plantvillage_dual_head_field.pt` (on Drive) - contains BOTH heads.

In [ ]:
DUAL_LAB_CKPT = CHECKPOINT_DIR / "best_plantvillage_dual_head_lab.pt"
assert DUAL_LAB_CKPT.exists(), f"Missing lab checkpoint: {DUAL_LAB_CKPT}. Run Step 4 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "resnet50",
    "--epochs", "10",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "field",
    "--init-from", str(DUAL_LAB_CKPT),
    "--tag", "dual_head_field",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head field training failed")
print("Done. Field head trained. Both heads now in one checkpoint.")

## Step 6 - Evaluate dual-head on BOTH test sets

The dual-head model runs both heads and picks the higher-confidence prediction
per sample. This should give strong numbers on BOTH domains.

In [ ]:
DUAL_FIELD_CKPT = CHECKPOINT_DIR / "best_plantdoc_dual_head_field.pt"
assert DUAL_FIELD_CKPT.exists(), f"Missing dual-head checkpoint: {DUAL_FIELD_CKPT}. Run Step 5 first."

for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(DUAL_FIELD_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--dual-head",
        "--variant", f"dual_head_resnet50_{dataset}",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ] + extra
    result = run(cmd, cwd=str(REPO_DIR), label=f"dual-head eval on {dataset} failed")
    print()

## Step 7 - The verdict: did dual-head break the trade-off?

**Sprint 8 gates (must BOTH pass):**
- PlantDoc F1 >= 0.60 (field gate)
- PlantVillage F1 >= 0.85 (lab gate)

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
keys = ["dual_head_resnet50_plantdoc", "dual_head_resnet50_plantvillage"]
dual_rows = df[df["variant"].isin(keys)].copy()
print(dual_rows[["variant", "f1"]].to_string(index=False))

if len(dual_rows) == 2:
    field_f1 = dual_rows[dual_rows["variant"] == "dual_head_resnet50_plantdoc"]["f1"].iloc[0]
    lab_f1 = dual_rows[dual_rows["variant"] == "dual_head_resnet50_plantvillage"]["f1"].iloc[0]
    field_pass = "PASS" if field_f1 >= 0.60 else "FAIL"
    lab_pass = "PASS" if lab_f1 >= 0.85 else "FAIL"
    print(f"\nDual-head verdict: field {field_f1:.4f} -> {field_pass} | lab {lab_f1:.4f} -> {lab_pass}")
else:
    print("\nDual-head rows not found. Run Step 6 first.")

## Where things live

**On Google Drive (durable):**
```
folium/checkpoints/best_plantvillage_dual_head_lab.pt     Stage 1: lab head trained
folium/checkpoints/best_plantdoc_dual_head_field.pt    Stage 2: field head trained
folium/results/ablation_results.csv                        All ablation rows (Sprint 7 + 8)
```

**Dual-head model:** one backbone + two independent heads. No catastrophic forgetting.
At inference, both heads run and the higher-confidence one wins per sample.